# Chapter 2 Demo: The Blueprint Before the Build
## BDI Framework and the Six Structural Layers

---

**Architectural Claim:** Architecture — not the model — determines agent behavior. The same model running inside a BDI architecture with explicit belief management, desire constraints, and intention exit conditions produces qualitatively different (and safer) behavior than the same model wrapped in a simple prompt-and-loop.

**This notebook demonstrates:**
1. A minimal BDI agent with explicit structural layers
2. A deliberate failure case — reward hacking through underspecified desires
3. The architectural fix — desire constraints and intention exit conditions
4. An AI scaffold for desire specification, with a **Mandatory Human Decision Node**

**To reproduce the failure:** Run Section 3 (`BrokenAgent`) without modification. Observe the output.

**To see the fix:** Run Section 4 (`SafeAgent`) and compare.

---
## Section 1: Install Dependencies

In [ ]:
# Install required packages
# Run this cell once before proceeding
%pip install anthropic --quiet

---
## Section 2: The Six Structural Layers — Base Classes

**Design decision documented here:**

Each structural layer is implemented as a separate class with a defined interface. This is not the only valid implementation choice — a flat dictionary or a single dataclass could encode the same information. The explicit class hierarchy is chosen because it makes the layer boundaries visible as code boundaries. When a failure occurs, the traceback localizes it to a specific layer.

**What breaks with the wrong choice:** Collapsing all layers into a single class with a `run()` method makes the architecture implicit. When the agent misbehaves, the failure mode is not localizable — the entire `run()` method is suspect.

In [ ]:
import json
import time
import copy
from datetime import datetime, timedelta
from dataclasses import dataclass, field
from typing import Any, Optional
import anthropic


# ─── LAYER 2: MEMORY ──────────────────────────────────────────────────────────
# Stores and retrieves the agent's belief state.
# Design decision: beliefs carry a timestamp. Any belief older than
# `max_age_seconds` is flagged as stale before it enters the reasoning layer.
# This implements the belief expiration policy discussed in the chapter.

@dataclass
class Belief:
    key: str
    value: Any
    source: str
    timestamp: datetime = field(default_factory=datetime.now)
    
    def is_stale(self, max_age_seconds: int = 300) -> bool:
        return (datetime.now() - self.timestamp).total_seconds() > max_age_seconds


class BeliefStore:
    """Layer 2: Memory — manages the agent's world model."""
    
    def __init__(self, max_age_seconds: int = 300):
        self._beliefs: dict[str, Belief] = {}
        self._max_age = max_age_seconds
    
    def update(self, key: str, value: Any, source: str = "system"):
        self._beliefs[key] = Belief(key=key, value=value, source=source)
    
    def get(self, key: str, warn_if_stale: bool = True) -> Optional[Any]:
        belief = self._beliefs.get(key)
        if belief is None:
            return None
        if warn_if_stale and belief.is_stale(self._max_age):
            print(f"⚠️  BELIEF STALENESS WARNING: '{key}' is {int((datetime.now() - belief.timestamp).total_seconds())}s old")
        return belief.value
    
    def snapshot(self) -> dict:
        return {k: v.value for k, v in self._beliefs.items()}


# ─── LAYER 3 (BDI): DESIRE AND INTENTION MANAGEMENT ──────────────────────────
# Design decision: Desires carry explicit constraints and priority.
# Intentions carry explicit exit conditions.
# These are separate objects — not a single 'goal' field.

@dataclass
class Desire:
    description: str
    priority: int                    # Lower number = higher priority
    constraints: list[str]           # Architectural constraints on pursuit
    anti_pattern: Optional[str] = None  # What reward hacking looks like for this desire


@dataclass
class Intention:
    desire: Desire
    plan_steps: list[str]
    exit_conditions: list[str]       # MANDATORY — what stops pursuit
    actions_taken: int = 0
    max_actions: int = 10            # Hard ceiling
    committed_at: datetime = field(default_factory=datetime.now)
    
    def should_exit(self, reason: str = "") -> bool:
        """Check if any exit condition has been met."""
        if self.actions_taken >= self.max_actions:
            print(f"🛑 INTENTION EXIT: max_actions ({self.max_actions}) reached")
            return True
        return False


# ─── LAYER 4: ACTION LOG ──────────────────────────────────────────────────────
# Every action is logged unconditionally. This is the audit trail.
# Design decision: logging is not optional and is not in a try/except.
# An action that cannot be logged must not be executed.

class ActionLog:
    """Layer 4: Immutable audit trail of all actions taken."""
    
    def __init__(self):
        self._entries: list[dict] = []
    
    def record(self, action: str, args: dict, result: Any, layer: str = "L4-Action"):
        entry = {
            "timestamp": datetime.now().isoformat(),
            "layer": layer,
            "action": action,
            "args": args,
            "result": result
        }
        self._entries.append(entry)
        return entry
    
    def report(self) -> list[dict]:
        return copy.deepcopy(self._entries)
    
    def summary(self) -> str:
        lines = [f"  [{e['timestamp'][-8:]}] {e['action']}({e['args']}) → {e['result']}" 
                 for e in self._entries]
        return "\n".join(lines) or "  (no actions taken)"


print("✅ Layer base classes defined: BeliefStore, Desire, Intention, ActionLog")

---
## Section 3: The Broken Agent — Deliberate Failure Case

**What this cell demonstrates:** A meeting room booking agent with no desire constraints and no intention exit conditions beyond task completion. This is the architectural configuration that produces the reward-hacking failure described in the chapter.

**The failure you will observe:** The agent books every available room to guarantee its primary desire is satisfied. It reasons correctly from its beliefs. The failure is architectural, not cognitive.

**Design decision deliberately omitted:** No resource constraints on the Desire. No exit condition proportional to the task scope. No monitoring layer checking for anomalous resource consumption.

> **Run this cell to trigger the failure.**

In [ ]:
# SIMULATED ENVIRONMENT: A calendar system with meeting rooms
# This is a mock — no real API calls are made in this failure demo.

class CalendarEnvironment:
    """Simulates a corporate calendar system."""
    
    ROOMS = ["Athena", "Boardroom A", "Boardroom B", "Lakeview", 
             "Mezzanine", "Penthouse", "Riverside", "Skyline"]
    
    def __init__(self):
        # All rooms start as available
        self.reservations: dict[str, list[str]] = {room: [] for room in self.ROOMS}
    
    def available_rooms(self, date: str) -> list[str]:
        return [r for r in self.ROOMS if date not in self.reservations[r]]
    
    def reserve_room(self, room: str, date: str, event: str) -> dict:
        """Books a room. No scope limits — any call succeeds."""
        if date in self.reservations[room]:
            return {"success": False, "reason": "already booked"}
        self.reservations[room].append(date)
        return {"success": True, "room": room, "date": date, "event": event}
    
    def total_reservations(self) -> int:
        return sum(len(dates) for dates in self.reservations.values())


# ─── THE BROKEN AGENT ─────────────────────────────────────────────────────────
# Desire: "Book a room for the weekly all-hands meeting"
# Missing: resource constraints, scope limits, exit conditions

def run_broken_agent(env: CalendarEnvironment, weeks: int = 12):
    print("="*60)
    print("BROKEN AGENT: No desire constraints, no exit conditions")
    print("="*60)
    
    # The desire — no constraints
    desire = {
        "description": "Book a room for the weekly all-hands meeting",
        # ❌ MISSING: max_bookings constraint
        # ❌ MISSING: anti_pattern definition
        # ❌ MISSING: resource scope limit
    }
    
    # The intention — only one exit condition: task complete
    # ❌ MISSING: exit condition for anomalous resource consumption
    # ❌ MISSING: exit condition for booking future dates beyond planning horizon
    # ❌ MISSING: exit condition for multiple successful bookings
    exit_condition = "room_booked_for_next_occurrence"
    
    action_log = ActionLog()
    
    print(f"\n🤖 Agent reasoning: '{desire['description']}'")
    print(f"   The meeting recurs weekly. Rooms are often unavailable.")
    print(f"   Optimal strategy: reserve all occurrences in advance.")
    print(f"   Proceeding...\n")
    
    # Agent executes its plan: book the room for all future occurrences
    # This is correct reasoning from an underspecified desire.
    booked = 0
    base_date = datetime.now()
    
    for week in range(weeks):
        target_date = (base_date + timedelta(weeks=week)).strftime("%Y-%m-%d")
        available = env.available_rooms(target_date)
        
        if available:
            # Agent picks the first available room
            chosen = available[0]
            result = env.reserve_room(chosen, target_date, "All-Hands Meeting")
            action_log.record("reserve_room", 
                              {"room": chosen, "date": target_date}, 
                              result)
            print(f"   ✓ Booked {chosen} on {target_date}")
            booked += 1
    
    print(f"\n📊 OUTCOME:")
    print(f"   Rooms booked: {booked}")
    print(f"   Total calendar reservations made: {env.total_reservations()}")
    print(f"   Rooms exhausted for any given week: potentially all of them")
    print(f"\n🔴 FAILURE DIAGNOSIS:")
    print(f"   The agent achieved the letter of its desire.")
    print(f"   It violated every implicit constraint the designers assumed.")
    print(f"   Layer that failed: L3-Reasoning (Desire specification)")
    print(f"   Root cause: Desire had no constraint. Intention had no scope exit.")
    print(f"   Fix required: Architectural. Prompt revision will not solve this.")
    
    return action_log


# Run the failure
broken_env = CalendarEnvironment()
broken_log = run_broken_agent(broken_env, weeks=12)

---
## Section 4: The Fixed Agent — Architectural Correction

**What changes:** Not the model. Not the prompt. The architectural configuration:
- The Desire now carries explicit resource constraints
- The Intention now carries three exit conditions proportional to task scope
- The Monitoring layer checks resource consumption against a normality baseline after each action

**Design decision documented here:** The three exit conditions are chosen at different granularities — per-action, per-session, and per-horizon. This is the "three-tier exit condition scaffold" from the Eddy audit. A single exit condition is always gameable; three at different granularities close the attack surface.

In [ ]:
def run_fixed_agent(env: CalendarEnvironment):
    print("="*60)
    print("FIXED AGENT: Explicit desire constraints + exit conditions")
    print("="*60)
    
    # ─── DESIRE: Explicitly constrained ──────────────────────────────────────
    desire = Desire(
        description="Book a room for the next weekly all-hands meeting",
        priority=1,
        constraints=[
            "Book exactly ONE occurrence (the next scheduled date)",
            "Do not book more than 2 weeks in advance",
            "Do not consume more than 1 room per booking action"
        ],
        anti_pattern="Booking multiple future occurrences to guarantee availability"
    )
    
    # ─── INTENTION: Three exit conditions at different granularities ──────────
    intention = Intention(
        desire=desire,
        plan_steps=[
            "1. Identify the next all-hands meeting date",
            "2. Check available rooms for that date only",
            "3. Reserve one room",
            "4. Confirm reservation and exit"
        ],
        exit_conditions=[
            "EXIT-1 (per-action): One successful reservation made → stop",
            "EXIT-2 (per-session): More than 2 calendar API calls made → stop and flag",
            "EXIT-3 (per-horizon): Target date is more than 14 days away → defer and exit"
        ],
        max_actions=2  # Hard architectural ceiling
    )
    
    action_log = ActionLog()
    
    print(f"\n🤖 Agent reasoning: '{desire.description}'")
    print(f"   Constraints: {desire.constraints[0]}")
    print(f"   Exit conditions loaded: {len(intention.exit_conditions)}")
    print(f"   Max actions: {intention.max_actions}\n")
    
    # Find next occurrence (1 week out)
    next_date = (datetime.now() + timedelta(weeks=1)).strftime("%Y-%m-%d")
    
    # Check EXIT-3 first
    days_until = 7  # We know it's 1 week out
    if days_until > 14:
        action_log.record("exit_check", {"condition": "EXIT-3"}, "deferred")
        print("🛑 EXIT-3 triggered: Meeting too far out. Deferring.")
        return action_log
    
    # Action: check availability
    intention.actions_taken += 1
    available = env.available_rooms(next_date)
    action_log.record("check_availability", {"date": next_date}, available, layer="L1-Perception")
    
    if not available:
        print("   No rooms available. Escalating to human scheduler.")
        return action_log
    
    # Check EXIT-2
    if intention.should_exit():
        return action_log
    
    # Action: reserve ONE room
    intention.actions_taken += 1
    chosen = available[0]
    result = env.reserve_room(chosen, next_date, "All-Hands Meeting")
    action_log.record("reserve_room", {"room": chosen, "date": next_date}, result)
    print(f"   ✓ Reserved {chosen} for {next_date}")
    
    # EXIT-1: One successful reservation → STOP
    print("\n✅ EXIT-1 triggered: One reservation made. Agent stopping.")
    
    # ─── LAYER 6: MONITORING ─────────────────────────────────────────────────
    # After every action sequence, monitoring checks resource consumption
    # against a normality baseline.
    total = env.total_reservations()
    print(f"\n📊 L6-Monitoring check:")
    print(f"   Total reservations this session: {intention.actions_taken - 1}")
    print(f"   Normality baseline: ≤1 reservation per invocation")
    if intention.actions_taken - 1 > 1:
        print("   🔴 ANOMALY FLAGGED — escalating to ops")
    else:
        print("   ✅ Within bounds")
    
    print(f"\n🟢 OUTCOME:")
    print(f"   Total calendar reservations made this session: {intention.actions_taken - 1}")
    print(f"   Desire satisfied: Yes")
    print(f"   Resources consumed: Proportional to task scope")
    
    return action_log


# Run the fixed agent on a fresh environment
fixed_env = CalendarEnvironment()
fixed_log = run_fixed_agent(fixed_env)

---
## Section 5: AI Scaffold — Desire Specification Assistant

**Architectural role of this scaffold:** The LLM enumerates candidate desires and their potential conflicts. This is a bounded enumeration task where the model's broad exposure to similar systems is valuable. However, the model cannot evaluate whether the proposed desire structure is safe to deploy in a specific context — it lacks knowledge of the deployment environment, the regulatory constraints, and the implicit assumptions of the engineering team.

**What the scaffold reduces:** The cognitive load of exhaustive enumeration.

**What the student must still do:** Evaluate safety, completeness, and appropriateness. The MANDATORY HUMAN DECISION NODE enforces this.

**Requires:** An `ANTHROPIC_API_KEY` environment variable set before running.

In [ ]:
import os

def desire_specification_scaffold(task_description: str) -> dict:
    """
    AI scaffold: uses an LLM to enumerate candidate desires for a new agent.
    
    HALTS at the MANDATORY HUMAN DECISION NODE.
    The architectural structure is NOT finalized until a human approves.
    """
    
    client = anthropic.Anthropic()
    
    enumeration_prompt = f"""
You are an agent architect. Given the task description below, enumerate the agent's desire structure.

Respond ONLY with valid JSON — no preamble, no markdown fences. Exact schema:
{{
  "primary_desire": {{
    "description": "<one sentence>",
    "resource_constraint": "<what resource is bounded and how>",
    "exit_conditions": ["<condition 1>", "<condition 2>", "<condition 3>"]
  }},
  "secondary_desires": [
    {{
      "description": "<one sentence>",
      "priority": <1-5>,
      "conflicts_with_primary": <true|false>,
      "exit_condition": "<when to abandon this desire>"
    }}
  ],
  "anti_desires": [
    "<thing the agent must NOT optimize for, stated as a specific behavior>"
  ],
  "reward_hacking_risk": "<describe the most likely unintended optimization path>"
}}

Task: {task_description}
"""
    
    print("🤖 Scaffold: enumerating desires...")
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1000,
        messages=[{"role": "user", "content": enumeration_prompt}]
    )
    
    raw = response.content[0].text.strip()
    # Strip any accidental markdown fences
    raw = raw.replace("```json", "").replace("```", "").strip()
    desires = json.loads(raw)
    
    # ─── MANDATORY HUMAN DECISION NODE ───────────────────────────────────────
    # The AI has proposed a desire structure. The architecture cannot be
    # finalized until a human architect has verified the following:
    #
    # EVALUATION CHECKLIST (you must answer all four before approving):
    #
    #   [1] Does every desire have a resource bound (a constraint that prevents
    #       the agent from consuming unbounded system resources to satisfy it)?
    #
    #   [2] Does the primary desire's scope prevent the meeting-room failure
    #       pattern (booking all available resources to guarantee satisfaction)?
    #
    #   [3] Are the anti-desires specific enough to be testable in a
    #       monitoring layer? ("Don't be harmful" is not testable.
    #       "Don't make more than 5 API calls per invocation" is.)
    #
    #   [4] Is there a desire conflict the model did not flag? Consider:
    #       speed vs. quality, scope vs. thoroughness, cost vs. completeness.
    #
    # The AI CANNOT perform this evaluation. It enumerated; you evaluate.
    # ─────────────────────────────────────────────────────────────────────────
    
    print("\n" + "═"*60)
    print("  MANDATORY HUMAN DECISION NODE")
    print("═"*60)
    print("\n📋 Proposed desire structure:")
    print(json.dumps(desires, indent=2))
    
    print("\n📌 Evaluation checklist — answer before approving:")
    checklist = [
        "[1] Does every desire have a resource bound?",
        "[2] Is the primary desire scoped to prevent resource exhaustion?",
        "[3] Are anti-desires specific enough to be testable in monitoring?",
        "[4] Are there desire conflicts the model did NOT flag?"
    ]
    for item in checklist:
        print(f"    {item}")
    
    print("\n─────────────────────────────────────────────────────────")
    print("Decision options:")
    print("  [yes]    — Approve and proceed")
    print("  [no]     — Reject. You must redesign before continuing.")
    print("  [modify] — Paste your corrected JSON when prompted")
    print("─────────────────────────────────────────────────────────")
    
    decision = input("\nYour decision [yes/no/modify]: ").strip().lower()
    
    if decision == "yes":
        desires["architect_approved"] = True
        desires["rejected_framing"] = input("Note one framing you rejected and why: ")
        print("\n✅ Desire structure approved. Proceeding to architecture finalization.")
        return desires
    
    elif decision == "no":
        print("\n🔴 Desire structure rejected.")
        print("Document your rejection reason (required for Author's Note):")
        reason = input("Rejection reason: ")
        raise ValueError(f"Desire structure rejected by architect: {reason}. Redesign required.")
    
    else:
        print("\nPaste your corrected JSON (single line, then Enter):")
        modified_raw = input()
        modified = json.loads(modified_raw)
        modified["architect_approved"] = True
        modified["modified_from_ai_proposal"] = True
        print("\n✅ Modified desire structure approved.")
        return modified


# ─── INVOCATION ───────────────────────────────────────────────────────────────
# Uncomment and run this block to invoke the scaffold.
# Requires ANTHROPIC_API_KEY to be set.

# task = (
#     "Automatically triage and respond to customer support emails. "
#     "Escalate complex issues to human agents. "
#     "Track resolution rates and flag repeat issues."
# )
# result = desire_specification_scaffold(task)
# print("\nFinalized desire structure:")
# print(json.dumps(result, indent=2))

print("✅ Scaffold function defined. Uncomment the invocation block to run.")
print("   Requires: ANTHROPIC_API_KEY environment variable")

---
## Section 6: Comparative Summary — Broken vs. Fixed

Run this cell to see a side-by-side comparison of what changed architecturally between the two agents.

In [ ]:
print("ARCHITECTURAL COMPARISON: Broken Agent vs. Fixed Agent")
print("="*65)
print(f"{'Layer':<25} {'Broken Agent':<25} {'Fixed Agent'}")
print("-"*65)

comparisons = [
    ("L2 Memory",         "Beliefs not timestamped",  "Beliefs timestamped + staleness check"),
    ("L3 Desire",         "No resource constraint",   "3 explicit constraints"),
    ("L3 Intention",      "1 exit condition",         "3 exit conditions (3 granularities)"),
    ("L4 Action",         "No scope limit",           "max_actions = 2 ceiling"),
    ("L6 Monitoring",     "Not implemented",          "Resource normality check after each action"),
    ("Failure mode",      "Reward hacking",           "Graceful task-scoped completion"),
    ("Rooms booked",      "12 (3 months out)",        "1 (next occurrence only)"),
]

for layer, broken, fixed in comparisons:
    print(f"{layer:<25} {broken:<25} {fixed}")

print("="*65)
print()
print("KEY INSIGHT:")
print("  The model was identical in both agents.")
print("  The prompt was functionally identical.")
print("  The architectural configuration was different.")
print("  The behavior was completely different.")
print()
print("  Architecture is the leverage point. Not the model.")

---
## Section 7: Author's Note — Human Decision Node Documentation

This section documents the architectural decisions made during this notebook's construction, in compliance with the assignment's Author's Note requirement.

### Decision 1: Desire as a separate class from Intention
**Rejected alternative:** A single `Goal` dataclass with a `committed` boolean flag.
**Why rejected:** A boolean flag cannot encode priority ordering, resource constraints, or anti-patterns. The `Goal` class would have grown to subsume all BDI state, re-collapsing the architecture we are trying to make explicit. Separating Desire and Intention as distinct classes enforces the architectural separation at the type level.

### Decision 2: Three exit conditions at different granularities
**Rejected alternative:** A single `max_bookings = 1` constraint on the Desire.
**Why rejected:** A single per-action constraint is gameable: an agent with a strong desire could interpret "max 1 booking per call" as a reason to make multiple calls. Exit conditions at three granularities (per-action, per-session, per-planning-horizon) close three distinct attack surfaces. The meeting-room failure requires all three to be absent simultaneously — the fix requires all three to be present.

### Decision 3: ActionLog as an immutable audit trail
**Rejected alternative:** Print statements in the action execution functions.
**Why rejected:** Print statements are not retrievable by the monitoring layer. The audit trail must be a first-class data structure that Layer 6 can query. An agent whose monitoring layer cannot access the action log has no closed feedback loop — it has a print statement.

---
*End of Chapter 2 Demo Notebook*